In [ ]:
import os
import re
import json
import open3d as o3d
import numpy as np
import CSF
from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt

def quat2rot(q):
    qw, qx, qy, qz = q['w_val'], q['x_val'], q['y_val'], q['z_val']
    return np.array([
        [1-2*qy*qy-2*qz*qz,   2*qx*qy-2*qz*qw,     2*qx*qz+2*qy*qw],
        [2*qx*qy+2*qz*qw,     1-2*qx*qx-2*qz*qz,   2*qy*qz-2*qx*qw],
        [2*qx*qz-2*qy*qw,     2*qy*qz+2*qx*qw,     1-2*qx*qx-2*qy*qy]
    ])

def get_pose(state):
    p = state['kinematics_estimated']['position']
    o = state['kinematics_estimated']['orientation']
    pos = np.array([p['x_val'], p['y_val'], p['z_val']], dtype=float)
    return pos, quat2rot(o)
import os, re, json
import numpy as np
from tqdm import tqdm
import open3d as o3d
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree
# ---------- labeling ----------
def compute_edge_labels(pcd: o3d.geometry.PointCloud, 
                        thresh: float = 3.0):
    xyz = np.asarray(pcd.points) 
    if xyz.size == 0: 
            return np.empty((0,), dtype=np.int32) 
    if center is None: 
        center = xyz.mean(axis=0) 
    center = np.asarray(center).reshape(1, 3) 
    dists = np.linalg.norm(xyz - center, axis=1) 
    labels = np.where(dists > 3.62, 50, 0).astype(np.float32) 
    return labels
def poisson_annulus_avoid(center_xy, r_inner, r_outer, existing_xy,
                          n_samples=400, min_dist=0.12,     # red–red spacing
                          obs_clear=0.18,                   # clearance from grey points
                          boundary_clear=0.03,              # keep off the orange lines
                          max_iter=100_000):
    """
    Poisson-disk-like sampling in an annulus that avoids existing points.
    - center_xy: (2,) ring center in world XY
    - existing_xy: (N,2) array of current grey points in world XY
    Returns (M,2) red points inside the empty space of the ring.
    """
    # keep only grey points near the ring (faster & more relevant)
    if existing_xy.size:
        d = np.hypot(existing_xy[:,0]-center_xy[0], existing_xy[:,1]-center_xy[1])
        mask = (d >= r_inner - obs_clear) & (d <= r_outer + obs_clear)
        obs = existing_xy[mask]
    else:
        obs = np.empty((0,2))

    tree = cKDTree(obs) if len(obs) > 0 else None

    red = []
    for _ in range(max_iter):
        th = np.random.uniform(0, 2*np.pi)
        r  = np.sqrt(np.random.uniform((r_inner+boundary_clear)**2,
                                       (r_outer-boundary_clear)**2))
        x = center_xy[0] + r*np.cos(th)
        y = center_xy[1] + r*np.sin(th)

        # keep away from grey points
        if len(obs):
            if tree is not None:
                dist, _ = tree.query([x, y])
                if dist < obs_clear:
                    continue
            else:
                if np.min((obs[:,0]-x)**2 + (obs[:,1]-y)**2) < obs_clear**2:
                    continue

        # Poisson spacing among red points
        if red:
            rr = np.asarray(red)
            if np.min((rr[:,0]-x)**2 + (rr[:,1]-y)**2) < min_dist**2:
                continue

        red.append((x, y))
        if len(red) >= n_samples:
            break

    return np.asarray(red) if red else np.zeros((0,2))
R_INNER = 1.7
R_OUTER = 3.50

# Poisson parameters (tweak to taste)
N_RED_POINTS = 300      # how many red points you want
MIN_DIST      = 0.01
# ---------- visualization colormap ----------
colors = [(0.5, 0.5, 0.5), (1, 1, 0), (1, 0, 0)]  # gray -> yellow -> red
custom_cmap = LinearSegmentedColormap.from_list("gray_yellow_red", colors, N=50)

data_root = "../data"

for track_id in range(1,8):
    track_folder = os.path.join(data_root, str(track_id))
    ply_files = sorted(
        [f for f in os.listdir(track_folder) if re.match(r"^\d+\.ply$", f)],
        key=lambda x: int(x.split('.')[0])
    )

    for ply_name in tqdm(ply_files, desc=f"Track {track_id}"):
        idx = int(ply_name.split('.')[0])

        # 1) load point cloud
        pcd = o3d.io.read_point_cloud(os.path.join(track_folder, ply_name))
        xyz = np.asarray(pcd.points)

        # 2) vehicle pose (position) for this frame
        car_state_path = os.path.join(track_folder, f"{idx}_car_state.json")
        with open(car_state_path, "r") as f:
            car_state = json.load(f)
        veh_pos, R = get_pose(car_state)  # veh_pos: shape (3,), R: 3x3

        existing_xy = xyz[:, :2]
        print(len(existing_xy))
        center = xyz.mean(axis=0)       # (3,)
        ring_center_xy = center[:2]     # (2,)
        red_xy = poisson_annulus_avoid(
            center_xy=ring_center_xy,
            r_inner=R_INNER,
            r_outer=R_OUTER,
            existing_xy=existing_xy,
            n_samples=N_RED_POINTS,
            min_dist=MIN_DIST,
            obs_clear=0.18,
            boundary_clear=0.05
        )
        # --- keep only "dense" red points (local density gate) ---
        LOCAL_R = 0.35          # radius to measure local density (tune)
        MIN_RED_NEIGHBORS = 6    # must have >= this many neighbors within LOCAL_R

        red_dense = np.empty((0,2))
        if len(red_xy):
            rtree = cKDTree(red_xy)
            neigh = rtree.query_ball_point(red_xy, r=LOCAL_R)
            # counts include self; subtract 1
            counts = np.array([len(ix)-1 for ix in neigh])
            red_dense = red_xy[counts >= MIN_RED_NEIGHBORS]

        # --- nearest grey points: only from dense red (and use k=1) ---
        nearest_idx = np.array([], dtype=int)
        if existing_xy.size and len(red_dense):
            gtree = cKDTree(existing_xy)
            _, idx_nn = gtree.query(red_dense, k=1)
            nearest_idx = np.unique(np.atleast_1d(idx_nn))

        nearest_xy = existing_xy[nearest_idx] if len(nearest_idx) else np.empty((0,2))

        # --- plotting (unchanged) ---
        fig, ax = plt.subplots(figsize=(5,5))
        if xyz.size:
            ax.scatter(xyz[:,0], xyz[:,1], s=6, marker='s', color='0.6', alpha=0.7, linewidths=0)

        circ_in  = plt.Circle(tuple(ring_center_xy), R_INNER, fill=False, color='tab:orange', lw=4)
        circ_out = plt.Circle(tuple(ring_center_xy), R_OUTER, fill=False, color='tab:orange', lw=4)
        ax.add_artist(circ_in); ax.add_artist(circ_out)

        if len(red_xy):
            ax.scatter(red_xy[:,0], red_xy[:,1], s=12, color='red', zorder=3)

        if len(nearest_xy):
            ax.scatter(nearest_xy[:,0], nearest_xy[:,1], s=12, color='green', zorder=4)

        ax.set_aspect('equal', 'box'); ax.set_xlabel('X'); ax.set_ylabel('Y')
        ax.set_title(f'Track {track_id} | Frame {idx}')
        plt.tight_layout()
        plt.show()

      


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
import os
import re
import json
import open3d as o3d
import numpy as np
import CSF
from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree

def quat2rot(q):
    qw, qx, qy, qz = q['w_val'], q['x_val'], q['y_val'], q['z_val']
    return np.array([
        [1-2*qy*qy-2*qz*qz,   2*qx*qy-2*qz*qw,     2*qx*qz+2*qy*qw],
        [2*qx*qy+2*qz*qw,     1-2*qx*qx-2*qz*qz,   2*qy*qz-2*qx*qw],
        [2*qx*qz-2*qy*qw,     2*qy*qz+2*qx*qw,     1-2*qx*qx-2*qy*qy]
    ])

def get_pose(state):
    p = state['kinematics_estimated']['position']
    o = state['kinematics_estimated']['orientation']
    pos = np.array([p['x_val'], p['y_val'], p['z_val']], dtype=float)
    return pos, quat2rot(o)
def poisson_annulus_avoid(center_xy, r_inner, r_outer, existing_xy,
                          n_samples=400, min_dist=0.12,     # red–red spacing
                          obs_clear=0.18,                   # clearance from grey points
                          boundary_clear=0.03,              # keep off the orange lines
                          max_iter=100_000):
    """
    Poisson-disk-like sampling in an annulus that avoids existing points.
    - center_xy: (2,) ring center in world XY
    - existing_xy: (N,2) array of current grey points in world XY
    Returns (M,2) red points inside the empty space of the ring.
    """
    # keep only grey points near the ring (faster & more relevant)
    if existing_xy.size:
        d = np.hypot(existing_xy[:,0]-center_xy[0], existing_xy[:,1]-center_xy[1])
        mask = (d >= r_inner - obs_clear) & (d <= r_outer + obs_clear)
        obs = existing_xy[mask]
    else:
        obs = np.empty((0,2))

    tree = cKDTree(obs) if len(obs) > 0 else None

    red = []
    for _ in range(max_iter):
        th = np.random.uniform(0, 2*np.pi)
        r  = np.sqrt(np.random.uniform((r_inner+boundary_clear)**2,
                                       (r_outer-boundary_clear)**2))
        x = center_xy[0] + r*np.cos(th)
        y = center_xy[1] + r*np.sin(th)

        # keep away from grey points
        if len(obs):
            if tree is not None:
                dist, _ = tree.query([x, y])
                if dist < obs_clear:
                    continue
            else:
                if np.min((obs[:,0]-x)**2 + (obs[:,1]-y)**2) < obs_clear**2:
                    continue

        # Poisson spacing among red points
        if red:
            rr = np.asarray(red)
            if np.min((rr[:,0]-x)**2 + (rr[:,1]-y)**2) < min_dist**2:
                continue

        red.append((x, y))
        if len(red) >= n_samples:
            break

    return np.asarray(red) if red else np.zeros((0,2))


def compute_annulus_gap_labels(
    pcd: o3d.geometry.PointCloud,
    *,
    center_xy: np.ndarray,           # (2,) – use veh_pos[:2]
    r_inner: float,
    r_outer: float,
    n_red_points: int = 300,
    min_dist: float = 0.12,          # red–red spacing
    obs_clear: float = 0.18,         # keep off grey points
    boundary_clear: float = 0.05,    # keep off circle borders
    local_r: float = 0.50,           # density radius for “enough red”
    min_red_neighbors: int = 6,      # neighbors within local_r (keep)
    min_fill_ratio: float = 0.40     # require this fraction of n_red_points to be dense (keep)
) -> np.ndarray:
    """
    Returns an (N,) int32 label array for pcd:
      50 for grey points within 0.2 m of sufficiently dense red fill inside the annulus,
       0 otherwise.
    """
    xyz = np.asarray(pcd.points)
    if xyz.size == 0:
        return np.array([], dtype=np.int32)

    existing_xy = xyz[:, :2]

    # 1) sample red fill points only in empty areas
    red_xy = poisson_annulus_avoid(
        center_xy, r_inner, r_outer, existing_xy,
        n_samples=n_red_points, min_dist=min_dist,
        obs_clear=obs_clear, boundary_clear=boundary_clear
    )

    if len(red_xy) == 0:
        return np.zeros(xyz.shape[0], dtype=np.int32)

    # 2) keep only locally-dense red points (prevents sparse false positives)
    rtree = cKDTree(red_xy)
    neigh = rtree.query_ball_point(red_xy, r=local_r)
    counts = np.array([len(ix) - 1 for ix in neigh])  # minus self
    red_dense = red_xy[counts >= min_red_neighbors]

    if len(red_dense) < max(1, int(min_fill_ratio * n_red_points)):
        return np.zeros(xyz.shape[0], dtype=np.int32)

    # 3) label ALL grey points within a fixed 0.2 m of any dense-red point
    label_radius = 0.6  # always
    gtree = cKDTree(existing_xy)
    idx_lists = gtree.query_ball_point(red_dense, r=label_radius)

    if len(idx_lists) == 0:  # no neighbors found
        return np.zeros(xyz.shape[0], dtype=np.int32)

    # Initialize as float to accumulate max risk, then cast to int32 at the end
    labels = np.zeros(xyz.shape[0], dtype=np.float32)

    # For each dense-red point, compute distances to its neighbors and update risk
    for red_pt, nbr_idx in zip(red_dense, idx_lists):
        if not nbr_idx:
            continue
        nbr_idx = np.asarray(nbr_idx, dtype=int)
        diffs = existing_xy[nbr_idx] - red_pt  # (k, 2)
        dists = np.linalg.norm(diffs, axis=1)  # (k,)

        # Linear ramp: 50 at d=0 down to 25 at d=label_radius
        risk = 50.0 - 25.0 * (dists / label_radius)
        # Clamp to [25, 50]
        np.clip(risk, 25.0, 50.0, out=risk)

        # Keep the maximum risk if multiple red points cover the same grey point
        np.maximum.at(labels, nbr_idx, risk)

    # Zero-out any points that never got labeled (risk stays 0)
    labels[labels < 25.0] = 0.0

    return labels.astype(np.int32)


R_INNER = 1.7
R_OUTER = 3.50

# Poisson parameters (tweak to taste)
N_RED_POINTS = 300      # how many red points you want
MIN_DIST      = 0.01

# ---------- visualization colormap ----------
colors = [(0.5, 0.5, 0.5), (1, 1, 0), (1, 0, 0)]  # gray -> yellow -> red
custom_cmap = LinearSegmentedColormap.from_list("gray_yellow_red", colors, N=50)

data_root = "../data"

for track_id in range(1, 8):
    track_folder = os.path.join(data_root, str(track_id))
    ply_files = sorted(
        [f for f in os.listdir(track_folder) if re.match(r"^\d+\.ply$", f)],
        key=lambda x: int(x.split('.')[0])
    )[80:]

    for ply_name in tqdm(ply_files, desc=f"Track {track_id}"):
        idx = int(ply_name.split('.')[0])

        # 1) load point cloud
        pcd = o3d.io.read_point_cloud(os.path.join(track_folder, ply_name))
        xyz = np.asarray(pcd.points)

        # 2) vehicle pose (position) for this frame
        car_state_path = os.path.join(track_folder, f"{idx}_car_state.json")
        with open(car_state_path, "r") as f:
            car_state = json.load(f)
        veh_pos, R = get_pose(car_state)  # veh_pos: shape (3,), R: 3x3
        center = xyz.mean(axis=0)      # (3,)
   
        # 3) compute labels (0 or 50) based on 3D distance to vehicle position
        labels25 = compute_annulus_gap_labels(
            pcd,
            center_xy=center[:2],   # use SAME center as your circles
            r_inner=R_INNER,
            r_outer=R_OUTER,
            n_red_points=N_RED_POINTS,
            min_dist=MIN_DIST,
            obs_clear=0.18,
            boundary_clear=0.05,
            local_r=0.35,
            min_red_neighbors=6,
            min_fill_ratio=0.40
        )
        
        norm = labels25.astype(np.float32) / 50.0          # [0,1]
        cols = custom_cmap(norm)[:, :3]  

        # 4) visualize
        vis_pcd = o3d.geometry.PointCloud()
        vis_pcd.points = o3d.utility.Vector3dVector(xyz)
        vis_pcd.colors = o3d.utility.Vector3dVector(cols)

        o3d.visualization.draw_geometries(
            [vis_pcd],
            window_name=f"Track {track_id} – Frame {idx}",
            width=800, height=600
        )

        # Optional: save labels
        # out_npy = os.path.join(track_folder, f"{idx}_labels_50_2.npy")
        # np.save(out_npy, combine_risk)

    #     break
    # break


Track 1:  97%|█████████▋| 64/66 [00:59<00:01,  1.08it/s]


KeyboardInterrupt: 